In [4]:
!pip install -q --upgrade pip

In [5]:
!pip install -q transformers datasets accelerate peft evaluate jiwer librosa soundfile

In [6]:
!pip install ijson

In [7]:
import os
import json
import random
import torch
import librosa
import numpy as np
from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

2025-12-15 16:39:14.832712: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765816754.856743     143 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765816754.864018     143 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [8]:
DATA_DIR = "/kaggle/input/weight-vietasr/train_full_manifest.jsonl"
MANIFEST_PATH = os.path.join(DATA_DIR)

def load_manifest(manifest_path):
    entries = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            entries.append({"audio_path": item["audio"], "sentence": item["text"]})
    print(f"✅ Loaded {len(entries)} samples")
    return entries

all_data = load_manifest(MANIFEST_PATH)

✅ Loaded 53997 samples


In [9]:
VAL_RATIO = 0.1
random.seed(42)
random.shuffle(all_data)
val_size = int(len(all_data) * VAL_RATIO)
train_entries = all_data[val_size:]
val_entries = all_data[:val_size]

print(f"Train: {len(train_entries)} samples")
print(f"Val: {len(val_entries)} samples")

Train: 48598 samples
Val: 5399 samples


In [10]:
MAX_LABEL_LEN = 448

def prepare_dataset(batch):
    audio_path = batch["audio_path"]

    try:
        audio_array, _ = librosa.load(audio_path, sr=16000)
    except:
        return None

    input_features = processor(
        audio_array,
        sampling_rate=16000
    ).input_features[0]

    labels = processor.tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=MAX_LABEL_LEN
    ).input_ids

    if len(labels) > MAX_LABEL_LEN:
        return None

    batch["input_features"] = input_features
    batch["labels"] = labels
    return batch

In [11]:
MODEL_DIR = "/kaggle/input/modell/other/modell/1/batch_10"

processor = WhisperProcessor.from_pretrained(MODEL_DIR)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="vi",
    task="transcribe"
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 512)
      (layers): ModuleList(
        (0-5): 6 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=False)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (fin

In [12]:
val_dataset = Dataset.from_list(val_entries)

# Map 1 lần duy nhất
val_dataset = val_dataset.map(
    prepare_dataset,
    remove_columns=val_dataset.column_names,
    desc="Preprocessing validation dataset"
)
print(f"✅ Validation dataset ready: {len(val_dataset)} samples")

Preprocessing validation dataset:   0%|          | 0/5399 [00:00<?, ? examples/s]

✅ Validation dataset ready: 5399 samples


In [13]:
from tqdm import tqdm

In [ ]:
test_dataset = "/kaggle/input/weight-vietasr/test_manifest.jsonl"
def load_test_manifest(test_dataset):
    entries = []
    with open(test_dataset, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            entries.append({
                "audio_path": item["audio"],
                "sentence": item["text"]
            })
    return entries
    
test_entries = load_test_manifest(test_dataset)

test_dataset = Dataset.from_list(test_entries)

test_dataset = test_dataset.map(
    prepare_dataset,
    remove_columns=test_dataset.column_names,
    desc="Preprocessing TEST dataset"
)

test_dataset = test_dataset.filter(lambda x: x is not None)

print(f"✅ Test samples: {len(test_dataset)}")

In [14]:
val_dataset.set_format(type="torch", columns=["input_features", "labels"])

wer_metric = evaluate.load("wer")
predictions, references = [], []

for sample in tqdm(val_dataset):
    input_features = sample["input_features"].unsqueeze(0).to(device)  # ✅ Giờ đã là tensor
    
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=256
        )
    
    pred_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]
    
    label_ids = sample["labels"]
    label_ids = [
        i if i != -100 else processor.tokenizer.pad_token_id
        for i in label_ids
    ]
    ref_text = processor.decode(label_ids, skip_special_tokens=True)
    
    def normalize(text):
        return text.lower().strip()

    predictions.append(normalize(pred_text))
    references.append(normalize(ref_text))

wer = wer_metric.compute(predictions=predictions, references=references)
print(f"🎯 FINAL WER Val: {wer * 100:.2f}%")

  0%|          | 0/5399 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 

🎯 FINAL WER Val: 43.01%


In [15]:
cer = evaluate.load("cer")
cer_score = cer.compute(predictions=predictions, references=references)
print("CER:", cer_score * 100)

CER: 32.026971816237335


In [16]:
for i in range(10):
    print("REF :", references[i])
    print("PRED:", predictions[i])
    print()

REF : anh vui lòng cho tôi tờ mười pound
PRED: anh vui lòng cho tôi tới tờ mười bao

REF : tôi bước ra khỏi giường , cầm điện thoại và vào phòng tắm .
PRED: tôi bước ra khỏi giữa cầm điện thoại và phòng tắm

REF : việc bị so sánh là bình thường , mỗi bài đều có số phận của nó rồi .
PRED: việc bị so sáng là bình thường mẫu bà điều có sức thận của nó rồi

REF : tuy nhiên , hôm qua anh để thua cả ba set .
PRED: tuy nhiên hôm qua anh em đã thuốc cả ba xe

REF : kỳ lạ thay những gì càng bí hiểm càng mông lung
PRED: khi lại thay những gì càng bí hiểm càng mong lung

REF : ta nghĩ mọi chuyện không đơn giản như vậy đâu
PRED: ta nghĩ mọi chuyện không đơn giản không như vậy đâu

REF : thị trường chứng khoán cần
PRED: thị trường chữ phán cần phú hí

REF : cháu hay thủ dâm lúc đang ngủ , làm sao để bỏ thói quen này ạ
PRED: tráo hề thủ dâm lúc đằng ngủ làm sao để bỏ thấu quen này ạ

REF : kêu gọi không đánh thuế xăng dầu trong ngày lễ
PRED: kêu gọi không đánh thuế xăng dầu trong ngày lễ

REF : hay 

In [ ]:
test_dataset.set_format(type="torch", columns=["input_features", "labels"])

wer_metric = evaluate.load("wer")
predictions, references = [], []

for sample in tqdm(test_dataset):
    input_features = sample["input_features"].unsqueeze(0).to(device)  # ✅ Giờ đã là tensor
    
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=256
        )
    
    pred_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]
    
    label_ids = sample["labels"]
    label_ids = [
        i if i != -100 else processor.tokenizer.pad_token_id
        for i in label_ids
    ]
    ref_text = processor.decode(label_ids, skip_special_tokens=True)
    
    predictions.append(pred_text)
    references.append(ref_text)

wer = wer_metric.compute(predictions=predictions, references=references)
print(f"🎯 FINAL WER Val: {wer * 100:.2f}%")